# 01 · LLM labels for Capesize news headlines

**What it does:** an *older* open LLM (Qwen2.5-7B-Instruct, released Sep 2024, so trained **before** our window from Aug 2025 → it cannot know what happened around the Hormuz closure) labels every headline as
**A Bullish · B Neutral · C Bearish · D Not relevant** for Capesize freight rates, using the same rules as `labelling_guide.md`.
It also creates the **100-headline check sheets** for Christella and Ninon.

**Before you start:** Runtime → Change runtime type → **T4 GPU**. Put `headlines_clean.parquet` in `My Drive/capesize_news/`.
**Run time:** about 30–60 min for ~9,500 headlines. Progress is saved to Drive every 500 headlines, so if Colab disconnects, just run all cells again and it continues.

**Leakage rules:** the prompt contains only the headline (and first paragraph): no dates, no question about what happened next.

In [ ]:
# 1. Connect Google Drive and point to the data folder
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
DATA = Path('/content/drive/MyDrive/capesize_news')   # <- your private Drive folder (do not share it: licensed Reuters text)
for f in ['headlines_clean.parquet']:
    print(f, 'OK' if (DATA / f).exists() else 'MISSING: upload it to ' + str(DATA))

In [ ]:
!pip -q install bitsandbytes accelerate openpyxl

In [ ]:
# 2. Load the merged headlines (from Ninon) and build the text the model sees
import pandas as pd, numpy as np
df = pd.read_parquet(DATA / 'headlines_clean.parquet')
df['lead'] = df['lead'].fillna('').str.strip()
df['has_lead'] = df['lead'].str.len() >= 40          # shorter "leads" are editorial notes or datelines
df['text_head'] = df['title'].fillna('').str.strip()
df['text_headlead'] = np.where(df['has_lead'], df['text_head'] + '. ' + df['lead'].str.slice(0, 300), df['text_head'])
df['period'] = np.where(pd.to_datetime(df['news_day']) < '2026-02-28', 'before closure', 'after closure')
print(len(df), 'headlines |', df['source'].value_counts().to_dict(), '| usable lead:', f"{df['has_lead'].mean():.0%}")

In [ ]:
# 3. The 100-headline check sheets (same 100 rows for both of you; label them independently, don't look at each other's)
#    50 before and 50 after the closure (28 Feb 2026), random but fixed (seed 42) so it can be reproduced.
from openpyxl import Workbook
from openpyxl.worksheet.datavalidation import DataValidation
from openpyxl.styles import Alignment, Font

check = (df.groupby('period', group_keys=False).apply(lambda g: g.sample(50, random_state=42))
           .sample(frac=1, random_state=42)[['id', 'news_day', 'source', 'title', 'lead']])
check['news_day'] = pd.to_datetime(check['news_day']).dt.strftime('%Y-%m-%d')

def write_sheet(path, who):
    wb = Workbook(); ws = wb.active; ws.title = 'label these'
    ws.append(['id', 'news_day', 'source', 'title', 'lead', 'label (A/B/C/D)', 'note (optional)'])
    for r in check.itertuples(index=False):
        ws.append([r.id, r.news_day, r.source, r.title, r.lead[:400], None, None])
    dv = DataValidation(type='list', formula1='"A,B,C,D"', allow_blank=True); ws.add_data_validation(dv)
    dv.add(f'F2:F{len(check) + 1}')
    for col, w in zip('ABCDEFG', [14, 12, 12, 70, 70, 16, 30]): ws.column_dimensions[col].width = w
    for row in ws.iter_rows(min_row=2):
        for c in row: c.alignment = Alignment(wrap_text=True, vertical='top')
    for c in ws[1]: c.font = Font(bold=True)
    ws.freeze_panes = 'A2'
    g = wb.create_sheet('labels'); [g.append(x) for x in [
        ['A', 'Bullish', 'Capesize rates likely UP'], ['B', 'Neutral', 'about dry bulk or its drivers, no clear direction'],
        ['C', 'Bearish', 'Capesize rates likely DOWN'], ['D', 'Not relevant', 'nothing to do with dry bulk or its cargoes'],
        [], ['See labelling_guide.md for the rules and examples. Use only the text, not what you know happened later.']]]
    g.column_dimensions['C'].width = 60
    wb.save(path); print('saved', path)

for who in ['christella', 'ninon']:
    p = DATA / f'check_100_{who}.xlsx'
    if p.exists(): print(p.name, 'already exists, not overwritten')
    else: write_sheet(p, who)

In [ ]:
# 4. Load the LLM in 4-bit (fits the free T4 GPU)
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
MODEL = 'Qwen/Qwen2.5-7B-Instruct'     # released Sep 2024: training data ends before our news window (Aug 2025 on)
tok = AutoTokenizer.from_pretrained(MODEL); tok.padding_side = 'left'
llm = AutoModelForCausalLM.from_pretrained(MODEL, device_map='auto',
        quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_compute_dtype=torch.float16))
llm.eval()
LETTERS = ['A', 'B', 'C', 'D']
LETTER_IDS = [tok.encode(l, add_special_tokens=False)[0] for l in LETTERS]
print('loaded', MODEL, '| letter token ids', LETTER_IDS)

In [ ]:
# 5. Instructions = the labelling guide in short (same rules the humans use)
RULES = """You label news headlines for a freight analyst. Question: if a Capesize ship owner read only this text, would they expect Capesize freight rates to go up or down over the next few weeks?
Capesize ships carry mainly iron ore (Brazil/Australia to China), coal and bauxite. Rates rise with more cargo, longer voyages or fewer available ships; they fall with less cargo, shorter voyages or more ships.
A = Bullish (rates likely up): more iron ore/coal/bauxite shipped or demanded, China stimulus or steel output up, ships rerouted on longer routes, port congestion or ships stuck, attacks or closures that slow shipping traffic or raise risk, capesize or Baltic index rising.
B = Neutral: about dry bulk or its drivers but no clear direction (mixed news, company deals, general fleet or regulation news, a tanker-only incident with no wider shipping effect).
C = Bearish (rates likely down): less iron ore/coal shipped or demanded, China steel output cuts or weak property sector, routes reopening or transits resuming, congestion easing, many new ships delivered, capesize or Baltic index falling.
D = Not relevant: nothing to do with dry bulk shipping or its cargoes (oil prices, tanker rates or container lines with no link to dry bulk, iron ore, coal or shipping routes; politics; sports; general markets).
Judge Capesize rates, not the general mood: bad news for the world can be good for freight rates. If unsure between a direction and B, answer B.
Examples:
"Baltic index hits one-week high as capesize rates surge" -> A
"Crew abandons burning tanker near Hormuz" -> A
"Iran says safe transit of Hormuz will be possible" -> C
"China orders steel output cuts to curb overcapacity" -> C
"Star Bulk announces strategic partnership with Hermes World Maritime" -> B
"Oil prices jump as OPEC+ extends supply cuts" -> D
Answer with one letter only: A, B, C or D."""

def prompt(text):
    msgs = [{'role': 'system', 'content': RULES}, {'role': 'user', 'content': f'Text: "{text}"\nLabel:'}]
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

print(prompt(df['text_headlead'].iloc[0])[-400:])

In [ ]:
# 6. Label every headline. One forward pass per headline: we read the probability of each letter (no free text).
#    Saved to Drive every 500 rows -> safe to re-run after a disconnect.
OUT = DATA / 'llm_labels.parquet'
done = pd.read_parquet(OUT) if OUT.exists() else pd.DataFrame()
todo = df[~df['id'].isin(done['id'])] if len(done) else df
print(f'{len(todo)} of {len(df)} headlines still to label')

BATCH = 16
rows = []
@torch.no_grad()
def label_batch(texts):
    enc = tok([prompt(t) for t in texts], return_tensors='pt', padding=True).to(llm.device)
    logits = llm(**enc).logits[:, -1, :][:, LETTER_IDS].float()
    return torch.softmax(logits, dim=-1).cpu().numpy()

import time; t0 = time.time()
for i in range(0, len(todo), BATCH):
    part = todo.iloc[i:i + BATCH]
    p = label_batch(part['text_headlead'].tolist())
    rows += [{'id': id_, 'p_bullish': a, 'p_neutral': b, 'p_bearish': c, 'p_notrel': d} for id_, (a, b, c, d) in zip(part['id'], p)]
    if len(rows) >= 500 or i + BATCH >= len(todo):
        done = pd.concat([done, pd.DataFrame(rows)], ignore_index=True); done.to_parquet(OUT, index=False); rows = []
        rate = (i + BATCH) / (time.time() - t0)
        print(f'{len(done)}/{len(df)} labelled | ~{(len(todo) - i - BATCH) / rate / 60:.0f} min left')

In [ ]:
# 7. Finish the file: label name, score (= P(bullish) - P(bearish)) and a quick look
lab = pd.read_parquet(OUT).drop_duplicates('id')
names = np.array(['bullish', 'neutral', 'bearish', 'not relevant'])
lab['llm_label'] = names[lab[['p_bullish', 'p_neutral', 'p_bearish', 'p_notrel']].to_numpy().argmax(1)]
lab['llm_score'] = lab['p_bullish'] - lab['p_bearish']
lab['llm_model'] = MODEL
lab.to_parquet(OUT, index=False)

m = df.merge(lab, on='id')
print(m['llm_label'].value_counts(normalize=True).round(3).to_string(), '\n')
print(pd.crosstab(m['period'], m['llm_label'], normalize='index').round(2), '\n')
for name in names:
    print(f'--- {name}:'); print(m.loc[m['llm_label'] == name, 'title'].sample(5, random_state=1).to_string(index=False))